# Equity Analysis Tool

Pulls live financial statements from Yahoo Finance, computes 14 fundamental ratios, runs a two-stage DCF with live country risk premiums from Damodaran and finally exports a PDF report, PowerPoint deck and CSV per ticker.

Supports any Yahoo Finance ticker. Non-USD financials are converted automatically at live exchange rates.

Run all cells top to bottom. Edit the tickers list and `use_ttm` flag in the last cell to customise according to the README instructions.

In [ ]:
!pip install -q yfinance pandas numpy matplotlib reportlab python-pptx

import yfinance as yf
import pandas as pd
import numpy as np
import os, math
import matplotlib.pyplot as plt
import requests

from io import StringIO
from datetime import datetime, timezone
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

from pptx import Presentation
from pptx.util import Inches
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor

plt.rcParams.update({'figure.max_open_warning': 0})

tickers = ["TSLA", "NVDA", "AAPL", "MSFT", "6301.T", "6501.T", "000333.SZ", "SAP.DE", "NESN.SW", "ASML.AS"]

## 1. Data extraction & formatting

In [ ]:
def get_row_label(df, possible_names):
    if df is None or df.empty:
        return None

    idx = [str(i).strip() for i in df.index.tolist()]

    for name in possible_names:
        if name in idx:
            return name

    lowered = [i.lower().replace(" ","").replace("_","").replace("&","") for i in idx]  # normalizing labels before comparing

    for name in possible_names:
        clean_name = name.lower().replace(" ","").replace("_","").replace("&","")
        if clean_name in lowered: return idx[lowered.index(clean_name)]
    return None



def extract_first_numeric(row):
    if isinstance(row, (int, float, np.number)):
        return float(row)
    for v in row.values:
        try:
            x = float(v)
            if not math.isnan(x):
                return x
        except: continue
    return np.nan



def safe_get(df, labels):
    lab = get_row_label(df, labels)
    if lab is None:
        return np.nan
    return extract_first_numeric(df.loc[lab])



def safe_get_ttm(df, labels):
    # Sum the most recent 4 quarterly values - correct TTM for income statement and cash flow rows.
    lab = get_row_label(df, labels)
    if lab is None:
        return np.nan
    row = df.loc[lab].dropna()
    vals = []
    for v in list(row.values)[:4]:
        try:
            x = float(v)
            if not math.isnan(x):
                vals.append(x)
        except Exception:
            continue
    return float(sum(vals)) if vals else np.nan



# label fields

revenue_labels = ["Total Revenue","Operating Revenue","Revenue","Net Sales","Sales"]
gross_profit_labels = ["Gross Profit","GrossProfit", "gross profit", "Gross profit", "Gross Operating Profit"]
net_income_labels = ["Net Income","NetIncome","Net Income From Continuing Operation Net Minority Interest","Net Income Applicable To Common Shares","NetIncomeLoss"]
total_assets_labels = ["Total Assets"]
ebitda_labels = ["EBITDA", "Ebitda", "Earnings Before Interest, Taxes, Depreciation And Amortization"]
equity_labels = ["Common Stock Equity","Stockholders Equity","Shareholders Equity","Total Equity Gross Minority Interest","Total Equity"]
debt_labels = ["Total Debt","Long Term Debt And Capital Lease Obligation","Long Term Debt","Current Debt"]
current_assets_labels = ["Total Current Assets","Current Assets"]
current_liab_labels = ["Total Current Liabilities","Current Liabilities"]
cash_broad_labels = ["Cash Cash Equivalents And Short Term Investments","Cash And Cash Equivalents And Short Term Investments","Cash, Cash Equivalents and Short-Term Investments"]
cash_narrow_labels = ["Cash And Cash Equivalents","Cash"]
ocf_labels = ["Operating Cash Flow","Total Cash From Operating Activities","Cash Flow From Continuing Operating Activities"]
capex_labels = ["Capital Expenditure","Capital Expenditures","Purchase Of PPE","Net PPE Purchase And Sale","Payments for Property, Plant and Equipment"]



# formatting

def fmt_money(v, currency="USD"):
    if np.isnan(v):
        return "n/a"
    negative = v < 0
    v = abs(v)
    symb = "$"
    if v >= 1e12: s = f"{v/1e12:.2f}T"
    elif v >= 1e9: s = f"{v/1e9:.2f}B"
    elif v >= 1e6: s = f"{v/1e6:.2f}M"
    else: s = f"{v:,.2f}"
    return ("-" + symb + s) if negative else symb + s



def format_value(k, v, currency="USD"):
    if np.isnan(v):
        return "n/a"

    k_lower = k.lower()

    # percentage-based ratios
    if k_lower in ["grossmargin", "netmargin", "roe", "roa", "upside"]:
        return f"{v:.1%}"

    # market multiples
    elif k_lower in ["pe", "ev_ebitda"]:
        return f"{v:.1f}×"

    # other ratios, excluding monetary ones
    elif k_lower in ["currentratio", "debttoequity", "cashratio", "cashflowreliability", "wacc", "netdebt_ebitda"]:
        return f"{v:.2f}"

    # monetary values & ratios that are amounts
    elif k_lower in ["netdebt", "fcf", "intrinsicvalue"] or k_lower in ["revenue", "grossprofit", "netincome", "ebitda", "totalassets", "totalequity", "totaldebt", "currentassets", "currentliabilities", "cash", "operatingcf", "capex", "marketcap"]:
        return fmt_money(v, currency)

    # all other float values not covered
    elif isinstance(v, float):
        return f"{v:.2f}"
    return str(v)



# Fetch the financial data from yfinance and convert to USD if applicable

def fetch_all(ticker, use_ttm=False):
    tk = yf.Ticker(ticker)

    if use_ttm:
        fin = tk.quarterly_financials if not tk.quarterly_financials.empty else None
        bs = tk.quarterly_balance_sheet if not tk.quarterly_balance_sheet.empty else None
        cf = tk.quarterly_cashflow if not tk.quarterly_cashflow.empty else None
    else:
        fin = tk.financials if not tk.financials.empty else None
        bs = tk.balance_sheet if not tk.balance_sheet.empty else None
        cf = tk.cashflow if not tk.cashflow.empty else None
    if fin is None or bs is None or cf is None:
        return {}

    # In TTM mode fin is quarterly. So we will fetch annual financials separately for use in charts and DCF CAGR, where we need multi-year annual revenue history, not individual quarters
    if use_ttm:
        annual_fin = tk.financials if not tk.financials.empty else fin
    else:
        annual_fin = fin

    out = {}

    sg = safe_get_ttm if use_ttm else safe_get                 # sum 4 quarters for TTM, single period for annual

    out["Revenue"]     = sg(fin, revenue_labels)
    out["GrossProfit"] = sg(fin, gross_profit_labels)
    out["NetIncome"]   = sg(fin, net_income_labels)
    out["EBITDA"] = sg(fin, ebitda_labels)
    out["TotalAssets"] = safe_get(bs, total_assets_labels)     # balance sheet = snapshot, always single period
    out["TotalEquity"] = safe_get(bs, equity_labels)
    out["TotalDebt"]   = safe_get(bs, debt_labels)
    out["CurrentAssets"]     = safe_get(bs, current_assets_labels)
    out["CurrentLiabilities"]= safe_get(bs, current_liab_labels)

    # BROAD CASH (includes short-term investments)
    broad = get_row_label(bs, cash_broad_labels)
    out["Cash"] = extract_first_numeric(bs.loc[broad]) if broad else safe_get(bs, cash_narrow_labels)

    capex_raw = sg(cf, capex_labels)
    out["Capex"] = abs(capex_raw) if not np.isnan(capex_raw) else np.nan

    out["OperatingCF"] = sg(cf, ocf_labels)

    info = tk.info

    out["marketCap"] = info.get("marketCap", np.nan)
    out["name"] = info.get("longName") or info.get("shortName", ticker)
    out["symbol"] = ticker
    out["currency"] = info.get("currency", "USD")
    out["beta"] = info.get("beta", 1.0)
    out["shares"] = info.get("sharesOutstanding", np.nan)
    out["currentPrice"] = info.get("currentPrice", np.nan)
    out["country"]      = info.get("country", "")

    # Optional USD conversion
    if out["currency"] != "USD":
        pair = f"USD{out['currency']}=X"
        rate_tk = yf.Ticker(pair)

        try:
            hist = rate_tk.history(period="5d")
            rate = float(hist["Close"].iloc[-1]) if not hist.empty else 1.0
        except Exception:
            rate = 1.0

        print(f"Converted {ticker} from {out['currency']} to USD at rate {rate}")

        # Convert the full dataframes to USD. Fix for mixed units in DCF and charts.
        if not fin.empty:
            fin = fin.apply(pd.to_numeric, errors='coerce') / rate
        if not bs.empty:
            bs = bs.apply(pd.to_numeric, errors='coerce') / rate
        if not cf.empty:
            cf = cf.apply(pd.to_numeric, errors='coerce') / rate
        if not annual_fin.empty:
            annual_fin = annual_fin.apply(pd.to_numeric, errors='coerce') / rate

        monetary_keys = [
            "Revenue", "GrossProfit", "NetIncome", "EBITDA",
            "TotalAssets", "TotalEquity", "TotalDebt",
            "CurrentAssets", "CurrentLiabilities", "Cash",
            "OperatingCF", "Capex", "marketCap", "currentPrice"
        ]

        for key in monetary_keys:
            if key in out and not np.isnan(out[key]):
                out[key] /= rate                                # dividing by local per USD to get USD value

        out["currency"] = "USD"

    out["fin_df"] = fin
    out["annual_fin_df"] = annual_fin                           # annual is used for multi-year revenue history in charts and DCF
    return out



def fetch_crp_table():

    # Damodaran's CRP page returns 403 so we spoof a browser header.
    # pandas can't auto-detect headers here (all <td>) so we scan the first rows manually

    url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ctryprem.html"
    try:
        resp = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0 Safari/537.36"},
            timeout=15
        )
        resp.raise_for_status()
        tables = pd.read_html(StringIO(resp.text), header=None)

        for tbl in tables:
            country_col = None
            crp_col     = None
            header_row  = None
            for row_index in range(min(5, len(tbl))):           # scan the first 5 rows to find which row is the header
                row_vals = [str(v).lower().strip() for v in tbl.iloc[row_index]]
                c_idx  = next((i for i, v in enumerate(row_vals) if v == "country"), None)
                crp_idx = next((i for i, v in enumerate(row_vals) if "country risk" in v), None)
                if c_idx is not None and crp_idx is not None:
                    country_col = c_idx
                    crp_col     = crp_idx
                    header_row  = row_index
                    break
            if country_col is None or crp_col is None:
                continue
            out = {}

            for row_index in range(header_row + 1, len(tbl)):
                country = str(tbl.iloc[row_index, country_col]).strip().lower()
                try:
                    raw_val = str(tbl.iloc[row_index, crp_col]).replace("%", "").strip()
                    crp_val = float(raw_val)
                    out[country] = crp_val / 100 if crp_val > 0.2 else crp_val
                except Exception:
                    continue
            if out:
                return out
    except Exception:
        pass
    return {}

## 2. Ratios & DCF

In [ ]:
# Ratios

def revenue_cagr(rev_hist, fallback=0.05):
    n = len(rev_hist) - 1

    if n > 0 and rev_hist[0] > 0 and rev_hist[-1] > 0:         # Avoid div-by-zero or negative growth issues
        raw = (rev_hist[-1] / rev_hist[0]) ** (1 / n) - 1
    else:
        raw = fallback                                         # Fallback approx long GDP growth or mature companies growth
    return max(min(raw, 0.15), 0.03)                           # Floor 3% long GDP and terminal growth minimum. Ceiling of 15% is used to avoid a hyper‑growth phase forever



def compute_ratios(data):
    r = {}
    r["GrossProfit"]            = data["GrossProfit"]
    r["GrossMargin"]            = data["GrossProfit"]/data["Revenue"] if data["Revenue"] else np.nan
    r["NetMargin"]              = data["NetIncome"]/data["Revenue"] if data["Revenue"] else np.nan
    r["ROA"]                    = data["NetIncome"]/data["TotalAssets"] if data["TotalAssets"] else np.nan
    r["ROE"]                    = data["NetIncome"]/data["TotalEquity"] if data["TotalEquity"] else np.nan
    r["CurrentRatio"]           = data["CurrentAssets"]/data["CurrentLiabilities"] if data["CurrentLiabilities"] else np.nan
    r["DebtToEquity"]           = data["TotalDebt"]/data["TotalEquity"] if data["TotalEquity"] else np.nan
    r["NetDebt"]                = data["TotalDebt"] - data["Cash"]
    r["CashRatio"]              = data["Cash"]/data["CurrentLiabilities"] if data["CurrentLiabilities"] else np.nan
    r["FCF"]                    = data["OperatingCF"] - data["Capex"] if not np.isnan(data["Capex"]) else np.nan
    r["CashflowReliability"]    = data["OperatingCF"]/data["NetIncome"] if data["NetIncome"] else np.nan
    r["PE"]                     = data["marketCap"] / data["NetIncome"] if data["NetIncome"] > 0 else np.nan
    enterprise_value            = data["marketCap"] + r["NetDebt"]
    r["EV_EBITDA"]              = enterprise_value / data["EBITDA"] if not np.isnan(data["EBITDA"]) and data["EBITDA"] != 0 else np.nan
    r["NetDebt_EBITDA"]         = r["NetDebt"] / data["EBITDA"] if not np.isnan(data["EBITDA"]) and data["EBITDA"] != 0 else np.nan
    return r



# DCF

# DCF valuation: 5-year FCF projection + terminal value, Gordon Growth Model
# Two-stage DCF: historical CAGR for the first 5 years, then terminal growth forever

def compute_dcf(raw, ratios, crp_table=None):

    # Assumptions
    risk_free       = 0.0419     # 10-year US treasury yield early 2026
    erp             = 0.05       # Equity Risk Premium, standard market
    cost_debt       = 0.05       # average corporate debt cost
    tax_rate        = 0.21       # US federal corporate tax
    terminal_growth = 0.03       # growth rate below GDP
    years = 5

    country = raw.get("country", "").lower()
    crp = (crp_table or {}).get(country, 0.0)                   # 0.0 fallback as developed markets
    beta = raw["beta"]                                          # from yfinance
    cost_equity = risk_free + beta * erp + crp

    mkt_eq = raw.get("marketCap", np.nan)
    market_equity = mkt_eq if (not np.isnan(mkt_eq) and mkt_eq > 0) else raw["TotalEquity"]
    total_capital = market_equity + raw["TotalDebt"]
    wacc = (market_equity / total_capital * cost_equity) + (raw["TotalDebt"] / total_capital * cost_debt * (1 - tax_rate))

    # Revenue history and forecast, two-stage
    rev_hist = []
    lbl = get_row_label(raw["annual_fin_df"], revenue_labels)

    if lbl:
        rev_hist = raw["annual_fin_df"].loc[lbl].dropna().astype(float).tolist()[::-1]
    if len(rev_hist) < 2:
        rev_hist = [0, 0]

    cagr = revenue_cagr(rev_hist)

    # For TTM mode use TTM revenue as base for consistency with FCF margin
    rev_base = raw["Revenue"] if not np.isnan(raw["Revenue"]) and raw["Revenue"] > 0 else rev_hist[-1]
    rev_forecast = [rev_base * (1 + cagr) ** (i + 1) for i in range(years)]

    # FCF projections
    current_fcf = ratios["FCF"]
    fcf_margin = current_fcf / raw["Revenue"] if raw["Revenue"] and raw["Revenue"] > 0 else 0.1   # fallback 10% if missing
    fcf_forecast = [rev * fcf_margin for rev in rev_forecast]                                     # constant-margin projection
    dcf_sum = sum(fcf / (1 + wacc) ** (i + 1) for i, fcf in enumerate(fcf_forecast))              # years 1-5 discount FCF

    # Terminal Value for Gordon Growth after year 5
    terminal_fcf = fcf_forecast[-1] * (1 + terminal_growth)
    if wacc <= terminal_growth:                                 # Gordon model breaks when g >= wacc
        terminal_value = 0
    else:
        terminal_value = terminal_fcf / (wacc - terminal_growth)
    terminal_pv = terminal_value / (1 + wacc) ** years

    enterprise_value = dcf_sum + terminal_pv
    equity_value = enterprise_value - ratios["NetDebt"]
    shares = raw["shares"]
    intrinsic_value = equity_value / shares if shares and not np.isnan(shares) and shares > 0 else np.nan

    current_price = raw["currentPrice"]
    upside = (intrinsic_value - current_price) / current_price if current_price and current_price > 0 else np.nan

    return {
        "WACC": wacc,
        "IntrinsicValue": intrinsic_value,
        "Upside": upside
    }

## 3. Commentary

In [ ]:
def make_commentary(raw, r, dcf):
    lines = []
    cf_rel = r["CashflowReliability"]

    if np.isnan(cf_rel):
        lines.append("Cash flow reliability: data unavailable.")
    elif cf_rel > 1.3:
        lines.append(f"Strong cash conversion: OCF is {cf_rel:.2f}× net income.")
    elif cf_rel > 1:
        lines.append(f"OCF exceeds net income ({cf_rel:.2f}×).")
    elif cf_rel > 0.7:
        lines.append(f"Moderate cash conversion (CF/NI = {cf_rel:.2f}), gap likely driven by working capital movements.")
    else:
        lines.append(f"Weak cash conversion (CF/NI = {cf_rel:.2f}).")

    lines.append(f"Profitability - ROE {r['ROE']:.1%} | ROA {r['ROA']:.1%} | Net margin {r['NetMargin']:.1%}")
    lines.append(f"Liquidity - Current ratio {r['CurrentRatio']:.2f} | Cash ratio {r['CashRatio']:.2f}")
    lines.append(f"Leverage - D/E {r['DebtToEquity']:.2f} | Net debt {fmt_money(r['NetDebt'], raw['currency'])}")
    lines.append(f"Free cash flow: {fmt_money(r['FCF'], raw['currency'])} (real, after {fmt_money(raw['Capex'], raw['currency'])} capex)")

    if not np.isnan(dcf["IntrinsicValue"]):
        upside_str = "overvalued" if dcf["Upside"] < 0 else "undervalued"
        lines.append(f"DCF Valuation: Intrinsic value {fmt_money(dcf['IntrinsicValue'], raw['currency'])}/share vs current {fmt_money(raw['currentPrice'], raw['currency'])} ({upside_str} by {abs(dcf['Upside']):.1%})")

    return "\n".join(lines)

## 4. Charts

In [ ]:
def make_charts(data, ratios, folder):
    os.makedirs(folder, exist_ok=True)



    # 1. Revenue forecast chart

    rev_hist_raw = []
    lbl = get_row_label(data["annual_fin_df"], revenue_labels)

    if lbl:
        rev_hist_raw = data["annual_fin_df"].loc[lbl].dropna().astype(float).tolist()[::-1]

    if len(rev_hist_raw) < 2:
        rev_hist_raw = [0, 0]

    # yfinance may not have the latest period's revenue row yet, common in the weeks after fiscal year close
    # We pin the fetched value so it appears as confirmed history rather than the first forecast point

    if lbl and not np.isnan(data["Revenue"]):
        try:
            latest_val = float(data["annual_fin_df"].loc[lbl, data["annual_fin_df"].columns.max()])
        except Exception:
            latest_val = np.nan
        if np.isnan(latest_val):
            rev_hist_raw.append(data["Revenue"])

    # Fallback to 5% if growth is zero/undefined
    cagr = revenue_cagr(rev_hist_raw)

    future_raw = [rev_hist_raw[-1] * (1 + cagr) ** (i + 1) for i in range(5)]

    rev_hist_plot = np.array(rev_hist_raw) / 1e9            # normalize plotting in billions
    future_plot = np.array(future_raw) / 1e9

    latest_date = data["annual_fin_df"].columns.max()
    base_year = pd.to_datetime(latest_date).year
    hist_years = list(range(base_year - len(rev_hist_plot) + 1, base_year + 1))
    forecast_years = list(range(base_year + 1, base_year + 6))
    bridge_years = [hist_years[-1]] + forecast_years       # overlap the last historical point so lines connect
    bridge_plot = np.concatenate([[rev_hist_plot[-1]], future_plot])

    plt.figure(figsize=(7,4))
    plt.plot(hist_years, rev_hist_plot, marker="o", label="History")
    plt.plot(bridge_years, bridge_plot, marker="o", linestyle="--", label="Forecast")
    plt.title("Revenue Forecast (5y CAGR Two-Stage)")
    plt.ylabel(f"Revenue ({data['currency']} Billion)")
    plt.xticks(rotation=45)
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{folder}/revenue_forecast.png", dpi=200); plt.close()



    # 2. Key metrics bar chart

    metrics = {k:v for k,v in ratios.items() if k in ["GrossMargin","NetMargin","ROE","ROA","CurrentRatio","CashRatio"] and not np.isnan(v)}
    if metrics:
        plt.figure(figsize=(7,4))
        bars = plt.bar(metrics.keys(), metrics.values(), color="teal")
        y_max = max(metrics.values()) * 1.1    # 10% headroom so labels fit inside chart
        plt.ylim(0, y_max)
        plt.title("Key Financial Ratios")
        plt.ylabel("Value (Ratios or %)")

        for label, bar in zip(metrics.keys(), bars):
            text_val = bar.get_height()
            if any(x in label for x in ["Margin", "ROE", "ROA"]):
                formatted_text = f"{text_val:.2%}"
            else:
                formatted_text = f"{text_val:.2f}"
            plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,formatted_text, ha="center")

        plt.grid(True, axis='y', alpha=0.3)
        plt.xticks(rotation=45); plt.tight_layout()
        plt.savefig(f"{folder}/key_ratios.png", dpi=200); plt.close()



    # 3. Cash flow chart

    plt.figure(figsize=(7, 4))
    net_income = data['NetIncome'] / 1e9
    ocf = data['OperatingCF'] / 1e9
    fcf_raw = ratios['FCF']
    fcf = fcf_raw / 1e9 if not np.isnan(fcf_raw) else np.nan

    cf_labels = ['Net Income', 'Operating CF', 'Free CF']
    cf_raw_values = [net_income, ocf, fcf]
    cf_plot_values = [0.0 if np.isnan(v) else v for v in cf_raw_values]
    cf_bars = plt.bar(cf_labels, cf_plot_values, color='teal')
    valid = [v for v in cf_raw_values if not np.isnan(v)]
    y_max = max(valid) * 1.1 if valid and max(valid) > 0 else 1
    y_min = min(valid) * 1.1 if valid and min(valid) < 0 else 0
    plt.ylim(y_min, y_max)

    plt.title('Cash Flow Bar Chart')
    plt.ylabel('Amount (USD Billion)')
    plt.grid(True, axis='y', alpha=0.3)

    for bar, raw_val in zip(cf_bars, cf_raw_values):
        height = bar.get_height()
        if np.isnan(raw_val):
            plt.text(bar.get_x() + bar.get_width() / 2, 0.05,
                     "n/a", ha='center', va='bottom', fontweight='bold', fontsize=11)
        else:
            if height >= 0:
                label_y = height + 0.15
                va = 'bottom'
            else:
                label_y = height - 0.15
                va = 'top'
            plt.text(bar.get_x() + bar.get_width() / 2, label_y,
                     f"{height:.1f}", ha='center', va=va, fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{folder}/cash_flow.png", dpi=200)
    plt.close()

## 5. PDF generation with summary, tables, charts

In [ ]:
def build_pdf(ticker, raw, ratios, commentary, folder, use_ttm=False, dcf=None):
    path = f"{folder}/{ticker}_FINAL.pdf"
    doc = SimpleDocTemplate(path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    year = "N/A"
    if not raw["fin_df"].empty:
        latest_date = raw["fin_df"].columns.max()
        year = pd.to_datetime(latest_date).year

    period = "TTM" if use_ttm else "Annual"
    year_str = f"{period} {year}"

    story.append(Paragraph(f"{ticker} — {raw['name']} — Fundamental Analysis Report ({year_str})", styles["Title"]))
    story.append(Paragraph(datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%SZ"), styles["Normal"]))
    story.append(Spacer(1,20))
    story.append(Paragraph(f"Executive Summary — {raw['name']}", styles["Heading2"]))
    story.append(Spacer(1, 8))

    for line in commentary.split("\n"):
        story.append(Paragraph(line, styles["BodyText"]))
        story.append(Spacer(1,6))

    story.append(Spacer(1,20))
    story.append(Paragraph("Key Metrics", styles["Heading2"]))
    data_table = [["Metric","Value"]]
    for k,v in ratios.items():
        data_table.append([k, format_value(k, v, raw["currency"])])
    table = Table(data_table, colWidths=[200,100])
    table.setStyle(TableStyle([('BACKGROUND', (0,0), (-1,0), colors.grey),
                               ('TEXTCOLOR',(0,0),(-1,0),colors.whitesmoke),
                               ('GRID', (0,0), (-1,-1), 0.5, colors.black)]))
    story.append(table)

    story.append(Spacer(1,20))
    story.append(PageBreak())
    story.append(Paragraph("DCF Valuation", styles["Heading2"]))
    dcf_table = [["Metric","Value"]]

    if dcf:
        for k,v in dcf.items():
            dcf_table.append([k, format_value(k, v, raw["currency"])])
    else:
        dcf_table.append(["Note", "DCF calculation unavailable"])
    dcf_tab = Table(dcf_table, colWidths=[200,100])
    dcf_tab.setStyle(TableStyle([('BACKGROUND', (0,0), (-1,0), colors.grey),
                                 ('TEXTCOLOR',(0,0),(-1,0),colors.whitesmoke),
                                 ('GRID', (0,0), (-1,-1), 0.5, colors.black)]))
    story.append(dcf_tab)

    # Charts
    for img in ["revenue_forecast.png","key_ratios.png","cash_flow.png"]:
        if os.path.exists(f"{folder}/{img}"):
            story.append(Spacer(1,20))
            story.append(Paragraph(img.replace(".png"," ").replace("_"," ").strip().title(), styles["Heading3"]))
            story.append(Image(f"{folder}/{img}", width=480, height=280))
            story.append(PageBreak())

    doc.build(story)
    return path

## 6. PPTX generation with summary, tables, charts

In [ ]:
def build_pptx(ticker, raw, ratios, commentary, folder, use_ttm=False, dcf=None):
    outpath = f"{folder}/{ticker}_FINAL.pptx"
    prs = Presentation()
    company_name = raw.get("name", ticker)
    period = "TTM" if use_ttm else "Annual"
    year = "N/A"
    if not raw["fin_df"].empty:
        latest_date = raw["fin_df"].columns.max()
        year = pd.to_datetime(latest_date).year
    year_str = f"{period} {year}"

    # Title Slide
    slide = prs.slides.add_slide(prs.slide_layouts[0])
    slide.shapes.title.text = f"{ticker} — {company_name} — Fundamental Analysis Report ({year_str})"
    subtitle = slide.placeholders[1]
    subtitle.text = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%SZ")

    # Executive Summary
    lines = [line for line in commentary.split("\n") if line.strip()]
    chunks = [lines[i:i+7] for i in range(0, len(lines), 7)]
    for i, chunk in enumerate(chunks):
        slide = prs.slides.add_slide(prs.slide_layouts[1])
        slide.shapes.title.text = "Executive Summary" if len(chunks) == 1 else f"Executive Summary (Part {i+1})"
        tf = slide.placeholders[1].text_frame
        tf.clear()
        tf.text = chunk[0]
        for line in chunk[1:]:
            p = tf.add_paragraph()
            p.text = line
            p.alignment = PP_ALIGN.LEFT

    # Key Metrics Table
    metrics_slide = prs.slides.add_slide(prs.slide_layouts[5])
    metrics_slide.shapes.title.text = "Key Metrics"
    rows, cols = len(ratios) + 1, 2
    left, top = Inches(1), Inches(1.5)
    width, height = Inches(8), Inches(0.4 * rows)
    table = metrics_slide.shapes.add_table(rows, cols, left, top, width, height).table
    table.cell(0, 0).text = "Metric"
    table.cell(0, 1).text = "Value"

    for col in range(cols):
        cell = table.cell(0, col)
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor(128, 128, 128)
        cell.text_frame.paragraphs[0].font.color.rgb = RGBColor(255, 255, 255)
    row_idx = 1
    for k, v in ratios.items():
        table.cell(row_idx, 0).text = k
        table.cell(row_idx, 1).text = format_value(k, v, raw["currency"])
        row_idx += 1

    # DCF Valuation Table
    dcf_slide = prs.slides.add_slide(prs.slide_layouts[5])
    dcf_slide.shapes.title.text = "DCF Valuation"
    dcf_rows = (len(dcf) + 1) if dcf else 2
    dcf_height = Inches(0.4 * dcf_rows)
    dcf_table = dcf_slide.shapes.add_table(dcf_rows, 2, left, top, width, dcf_height).table
    dcf_table.cell(0, 0).text = "Metric"
    dcf_table.cell(0, 1).text = "Value"
    for col in range(2):
        cell = dcf_table.cell(0, col)
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor(128, 128, 128)
        cell.text_frame.paragraphs[0].font.color.rgb = RGBColor(255, 255, 255)
    if dcf:
        row_idx = 1
        for k, v in dcf.items():
            dcf_table.cell(row_idx, 0).text = k
            dcf_table.cell(row_idx, 1).text = format_value(k, v, raw["currency"])
            row_idx += 1
    else:
        dcf_table.cell(1, 0).text = "Note"
        dcf_table.cell(1, 1).text = "DCF calculation unavailable"

    # Charts
    imgs = ["revenue_forecast.png", "key_ratios.png", "cash_flow.png"]
    for img in imgs:
        img_path = f"{folder}/{img}"
        if os.path.exists(img_path):
            slide = prs.slides.add_slide(prs.slide_layouts[5])
            slide.shapes.title.text = img.replace(".png", "").replace("_", " ").title()
            slide.shapes.add_picture(img_path, Inches(1), Inches(1.5), height=Inches(4.5))

    prs.save(outpath)
    return outpath

## 7. Main runner loop

In [ ]:
def run_everything(tickers=["TSLA"], outdir="FINAL_VERSION", use_ttm=False):
    os.makedirs(outdir, exist_ok=True)
    print("Fetching Damodaran CRP table...")
    crp_table = fetch_crp_table()
    print(f"{len(crp_table)} countries loaded." if crp_table else "CRP fetch failed, defaulting to 0% for all markets.")

    for t in tickers:
        print(f"{t}")
        try:
            raw = fetch_all(t, use_ttm=use_ttm)
            if not raw:
                print(f"Skipped {t}: no data returned (try use_ttm=False if currently True)")
                continue
            ratios = compute_ratios(raw)
            dcf = compute_dcf(raw, ratios, crp_table=crp_table)
            comm = make_commentary(raw, ratios, dcf)
            folder = f"{outdir}/{t}"
            os.makedirs(folder, exist_ok=True)
            make_charts(raw, ratios, folder)
            build_pdf(t, raw, ratios, comm, folder, use_ttm=use_ttm, dcf=dcf)
            build_pptx(t, raw, ratios, comm, folder, use_ttm=use_ttm, dcf=dcf)
            pd.DataFrame([ratios]).to_csv(f"{folder}/{t}_metrics.csv", index=False)
            print(f"Done: {folder}/")
        except Exception as e:
            print(f"Failed {t}: {e}")
            continue

    print(f"Finished! All files in {outdir}")

run_everything(tickers, "Financial_Results")